## Part 1: Preprocessing

In [20]:
# Import our dependencies
! pip install scikit-learn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Concatenate

#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [21]:
# Determine the number of unique values in each column
attrition_df.nunique()

Age                         43
Attrition                    2
BusinessTravel               3
Department                   3
DistanceFromHome            29
Education                    5
EducationField               6
EnvironmentSatisfaction      4
HourlyRate                  71
JobInvolvement               4
JobLevel                     5
JobRole                      9
JobSatisfaction              4
MaritalStatus                3
NumCompaniesWorked          10
OverTime                     2
PercentSalaryHike           15
PerformanceRating            2
RelationshipSatisfaction     4
StockOptionLevel             4
TotalWorkingYears           40
TrainingTimesLastYear        7
WorkLifeBalance              4
YearsAtCompany              37
YearsInCurrentRole          19
YearsSinceLastPromotion     16
YearsWithCurrManager        18
dtype: int64

In [22]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']].copy()

In [23]:
# Create a list of at least 10 column names to use as X data
x_columns = [
    'Age',
    'DistanceFromHome',
    'JobLevel',
    'HourlyRate',
    'JobRole',
    'OverTime',
    'YearsAtCompany',
    'TotalWorkingYears',
    'EducationField',
    'WorkLifeBalance',
    'Department'
]

# Create X_df using your selected columns
x_df = attrition_df[x_columns].copy()

# Show the data types for X_df
print(x_df.dtypes)

Age                   int64
DistanceFromHome      int64
JobLevel              int64
HourlyRate            int64
JobRole              object
OverTime             object
YearsAtCompany        int64
TotalWorkingYears     int64
EducationField       object
WorkLifeBalance       int64
Department           object
dtype: object


In [24]:
# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    x_df, y_df['Attrition'], test_size=0.2, random_state=42, stratify=y_df['Attrition']
)
y_train = y_train.map({'Yes': 1, 'No': 0}).astype(np.float32)
y_test = y_test.map({'Yes': 1, 'No': 0}).astype(np.float32)

In [25]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
# Convert your X data to numeric data types however you see fit
categorical_cols = X_train_raw.select_dtypes(include='object').columns.tolist()
numerical_cols = X_train_raw.select_dtypes(exclude='object').columns.tolist()

# Create a StandardScaler
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Fit the StandardScaler to the training data
X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")


X_train shape: (1176, 27)
X_test shape: (294, 27)


In [26]:
from sklearn.preprocessing import OneHotEncoder

# Create a OneHotEncoder for the Department column
dept_train = X_train_raw[['Department']]
dept_test = X_test_raw[['Department']]
dept_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit the encoder to the training data
dept_encoder.fit(dept_train)


# Create two new variables by applying the encoder
# to the training and testing data
dept_train_encoded = dept_encoder.transform(dept_train)
dept_test_encoded = dept_encoder.transform(dept_test)

In [27]:
# Create a OneHotEncoder for the Attrition column
attr_train = y_train.to_frame(name='Attrition')
attr_test = y_test.to_frame(name='Attrition')
attr_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


# Fit the encoder to the training data
attr_encoder.fit(attr_train)

# Create two new variables by applying the encoder
# to the training and testing data
attr_train_encoded = attr_encoder.transform(attr_train)
attr_test_encoded = attr_encoder.transform(attr_test)

## Part 2: Create, Compile, and Train the Model

In [28]:
# Find the number of columns in the X training data.
input_dim_main = X_train.shape[1]
input_dim_dept = dept_train_encoded.shape[1]
input_dim_attr = attr_train_encoded.shape[1]


# Create the input layer
main_input = Input(shape=(input_dim_main,), name='main_input')
dept_input = Input(shape=(input_dim_dept,), name='dept_input')
attr_input = Input(shape=(input_dim_attr,), name='attr_input')

# Create at least two shared layers
shared = Dense(64, activation='relu')(main_input)
shared = Dense(32, activation='relu')(shared)

In [29]:
# Create a branch for Department and Attrition
# with a hidden layer and an output layer
dept_hidden = Dense(16, activation='relu')(dept_input)
attr_hidden = Dense(8, activation='relu')(attr_input)
merged = Concatenate()([shared, dept_hidden, attr_hidden])
# Create the hidden layer
hidden = Dense(16, activation='relu')(merged)

# Create the output layer
output = Dense(1, activation='sigmoid', name='output')(hidden)

In [30]:
# Create the model
model = Model(inputs=[main_input, dept_input, attr_input], outputs=output)


# Compile the model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# Summarize the model
model.summary()

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 main_input (InputLayer)        [(None, 27)]         0           []                               
                                                                                                  
 dense_5 (Dense)                (None, 64)           1792        ['main_input[0][0]']             
                                                                                                  
 dept_input (InputLayer)        [(None, 3)]          0           []                               
                                                                                                  
 attr_input (InputLayer)        [(None, 2)]          0           []                               
                                                                                            

In [31]:
# Train the model
model.fit(
    [X_train, dept_train_encoded, attr_train_encoded],
    y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/20
30/30 [==============================] - 1s 13ms/step - loss: 0.4537 - accuracy: 0.8755 - val_loss: 0.3453 - val_accuracy: 0.8432
Epoch 2/20
30/30 [==============================] - 0s 4ms/step - loss: 0.2850 - accuracy: 0.8596 - val_loss: 0.2392 - val_accuracy: 0.8856
Epoch 3/20
30/30 [==============================] - 0s 3ms/step - loss: 0.1982 - accuracy: 0.9340 - val_loss: 0.1655 - val_accuracy: 0.9576
Epoch 4/20
30/30 [==============================] - 0s 4ms/step - loss: 0.1318 - accuracy: 0.9649 - val_loss: 0.1062 - val_accuracy: 0.9831
Epoch 5/20
30/30 [==============================] - 0s 4ms/step - loss: 0.0808 - accuracy: 0.9968 - val_loss: 0.0638 - val_accuracy: 1.0000
Epoch 6/20
30/30 [==============================] - 0s 4ms/step - loss: 0.0474 - accuracy: 1.0000 - val_loss: 0.0386 - val_accuracy: 1.0000
Epoch 7/20
30/30 [==============================] - 0s 5ms/step - loss: 0.0293 - accuracy: 1.0000 - val_loss: 0.0250 - val_accuracy: 1.0000
Epoch 8/20
30/30 [=

In [34]:
# Evaluate the model with the testing data
loss, accuracy = model.evaluate(
    [X_test, dept_test_encoded, attr_test_encoded],
    y_test,
    verbose=1
)

10/10 [==============================] - 0s 2ms/step - loss: 0.0021 - accuracy: 1.0000


In [35]:
# Print the accuracy for both department and attrition
print(f"\nTest Accuracy: {accuracy:.4f}")


Test Accuracy: 1.0000


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. Accuracy does not appear to be the best metric. It returned a score of 1 which doesn't seem possible.
2. I used the Relu activiation function.
3. I could alter the epochs maybe.